# Merge Soil and Meteorological Data
## Overview
This notebook merges cleaned soil and meteorological (MET) data by **aligning timestamps**.  
The merged dataset will be used for further analysis and modeling.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
def load_data(station_id=1):
    file_path_soil = Path(f"../EDA_2025/cleaned_soil_data/SM_{station_id}_cleaned.csv")
    file_path_met = Path(f"../EDA_2025/cleaned_met_data/MET_{station_id}_cleaned.csv")
    df_soil = pd.read_csv(file_path_soil, parse_dates=["Date"], index_col="Date")
    df_met = pd.read_csv(file_path_met, parse_dates=["Date"], index_col="Date")
    return df_soil, df_met

In [3]:
df_soil, df_met = load_data(station_id=6)

# Preview loaded data
print("Soil Data:")
print(df_soil.head())
print("\nMET Data:")
print(df_met.head())

Soil Data:
                     Ppt  SWC_5  SWC_10  SWC_20  SWC_50   T_5  T_10  T_20  \
Date                                                                        
2015-01-01 00:00:00  0.0  0.116   0.123   0.137   0.084  3.85  4.64  5.82   
2015-01-01 01:00:00  0.0  0.116   0.122   0.137   0.085  3.84  4.62  5.78   
2015-01-01 02:00:00  0.0  0.116   0.122   0.137   0.085  3.83  4.59  5.74   
2015-01-01 03:00:00  0.0  0.115   0.122   0.137   0.085  3.84  4.57  5.69   
2015-01-01 04:00:00  0.0  0.115   0.123   0.137   0.085  3.84  4.55  5.64   

                     T_50  Flag  
Date                             
2015-01-01 00:00:00  9.58     0  
2015-01-01 01:00:00  9.52   768  
2015-01-01 02:00:00  9.47   768  
2015-01-01 03:00:00  9.43   768  
2015-01-01 04:00:00  9.39     0  

MET Data:
                     Ppt   Tair    RH  Wind speed  Wind direction  Srad
Date                                                                   
2015-01-01 00:00:00  0.0 -1.013  82.4       0.995       

In [4]:
print(df_soil.shape)
print(df_met.shape)

(59161, 10)
(58441, 6)


In [5]:
def merge_datasets(df_soil, df_met):
    """
    Merge soil and meteorological data
    
    Args:
        df_soil: Soil dataset with DateTime index
        df_met: Meteorological dataset with DateTime index
    
    Returns:
        Merged DataFrame with aligned timestamps
    """
    
    # Inner join to keep only matching timestamps
    merged_df = pd.merge(
        df_soil,
        df_met,
        how='inner',  
        left_index=True,
        right_index=True,
        suffixes=('_soil', '_met')
    )
    
    # Validate merge quality
    print(f"Merged dataset contains {len(merged_df)} hourly records")
    print(f"Time range: {merged_df.index.min()} to {merged_df.index.max()}")
    
    return merged_df.sort_index()

In [6]:
merged_df = merge_datasets(df_soil, df_met)

Merged dataset contains 58441 hourly records
Time range: 2015-01-01 00:00:00 to 2021-09-01 00:00:00


In [7]:
merged_df.head(5)

,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,,,,,,,,,,,
2015-01-01 00:00:00,0.0,0.116,0.123,0.137,0.084,3.85,4.64,5.82,9.58,0,0.0,-1.013,82.4,0.995,36.94,0.00
2015-01-01 01:00:00,0.0,0.116,0.122,0.137,0.085,3.84,4.62,5.78,9.52,768,0.0,-0.981,83.3,0.857,30.15,0.56
2015-01-01 02:00:00,0.0,0.116,0.122,0.137,0.085,3.83,4.59,5.74,9.47,768,0.0,-0.910,83.9,1.040,44.80,0.00
2015-01-01 03:00:00,0.0,0.115,0.122,0.137,0.085,3.84,4.57,5.69,9.43,768,0.0,-0.842,82.6,0.936,50.38,0.00
2015-01-01 04:00:00,0.0,0.115,0.123,0.137,0.085,3.84,4.55,5.64,9.39,0,0.0,-0.785,89.9,0.606,358.20,0.00


In [8]:
merged_df.tail(5)

,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,,,,,,,,,,,
2021-08-31 20:00:00,0.0,0.104,0.101,0.09,0.062,35.04,34.77,33.60,30.74,0,0.0,31.41,100.00,0.122,181.6,0.02
2021-08-31 21:00:00,0.0,0.104,0.100,0.09,0.062,34.52,34.46,33.64,30.80,0,0.0,30.31,100.00,0.013,184.3,0.00
2021-08-31 22:00:00,0.0,0.103,0.100,0.09,0.062,33.99,34.07,33.58,30.86,0,0.0,29.99,2.55,0.309,168.6,0.00
2021-08-31 23:00:00,0.0,0.103,0.100,0.09,0.062,33.47,33.67,33.45,30.93,0,0.0,29.45,100.00,1.252,172.7,0.00
2021-09-01 00:00:00,0.0,0.103,0.100,0.09,0.062,33.00,33.30,33.27,31.01,0,0.0,28.70,83.20,1.524,180.2,0.00


In [9]:
print("\nColumn Summary:")
merged_df.dtypes


Column Summary:


Ppt_soil          float64
SWC_5             float64
SWC_10            float64
SWC_20            float64
SWC_50            float64
T_5               float64
T_10              float64
T_20              float64
T_50              float64
Flag                int64
Ppt_met           float64
Tair              float64
RH                float64
Wind speed        float64
Wind direction    float64
Srad              float64
dtype: object

In [10]:
merged_df.describe()

,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
count,58438.000000,58438.000000,58437.000000,58437.000000,58436.000000,58438.000000,58438.000000,58438.000000,58438.000000,58441.000000,58439.000000,58439.000000,58439.000000,58439.000000,58439.000000,58343.000000
mean,0.070490,0.113998,0.117997,0.126012,0.083402,22.761897,22.863614,22.839548,22.664643,25.032819,0.070530,19.243774,65.460541,1.904108,178.106064,204.592618
std,0.830763,0.031217,0.027264,0.028210,0.016951,9.409473,9.184646,8.812416,7.768793,399.606605,0.830765,9.150696,26.425602,1.341123,90.251473,293.666419
min,0.000000,0.051000,0.058000,0.070000,0.050000,1.470000,2.080000,2.900000,5.180000,0.000000,0.000000,-15.900000,1.090000,0.000000,0.000000,0.000000
25%,0.000000,0.090000,0.096000,0.101000,0.072000,14.990000,15.180000,15.280000,15.522500,0.000000,0.000000,13.170000,47.075000,0.836000,134.800000,0.000000
50%,0.000000,0.113000,0.118000,0.130000,0.086000,23.060000,23.240000,23.310000,22.920000,0.000000,0.000000,20.670000,70.910000,1.848000,178.500000,8.730000
75%,0.000000,0.136000,0.138000,0.148000,0.096000,30.420000,30.510000,30.460000,29.880000,0.000000,0.000000,25.680000,87.800000,2.756000,235.100000,366.300000
max,46.230000,0.253000,0.247000,0.235000,0.153000,45.640000,43.540000,40.750000,36.800000,13107.000000,46.230000,41.700000,100.000000,9.480000,360.000000,1110.000000


In [11]:
def save_merged_data(df, station_id, output_dir="merged_data"):
    """
    Save merged data to CSV.
    
    Args:
        df (pd.DataFrame): Merged soil and met data.
        station_id (int): Station ID (1-6).
        output_dir (str): Output directory path.
    """
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/merged_station_{station_id}.csv"
    df.to_csv(output_path)
    print(f"Saved merged data to: {output_path}")

    return df

In [12]:
#save_merged_data(merged_df, 6)

Saved merged data to: merged_data/merged_station_6.csv


,Ppt_soil,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_met,Tair,RH,Wind speed,Wind direction,Srad
Date,,,,,,,,,,,,,,,,
2015-01-01 00:00:00,0.0,0.116,0.123,0.137,0.084,3.85,4.64,5.82,9.58,0,0.0,-1.013,82.40,0.995,36.94,0.00
2015-01-01 01:00:00,0.0,0.116,0.122,0.137,0.085,3.84,4.62,5.78,9.52,768,0.0,-0.981,83.30,0.857,30.15,0.56
2015-01-01 02:00:00,0.0,0.116,0.122,0.137,0.085,3.83,4.59,5.74,9.47,768,0.0,-0.910,83.90,1.040,44.80,0.00
2015-01-01 03:00:00,0.0,0.115,0.122,0.137,0.085,3.84,4.57,5.69,9.43,768,0.0,-0.842,82.60,0.936,50.38,0.00
2015-01-01 04:00:00,0.0,0.115,0.123,0.137,0.085,3.84,4.55,5.64,9.39,0,0.0,-0.785,89.90,0.606,358.20,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-31 20:00:00,0.0,0.104,0.101,0.090,0.062,35.04,34.77,33.60,30.74,0,0.0,31.410,100.00,0.122,181.60,0.02
2021-08-31 21:00:00,0.0,0.104,0.100,0.090,0.062,34.52,34.46,33.64,30.80,0,0.0,30.310,100.00,0.013,184.30,0.00
2021-08-31 22:00:00,0.0,0.103,0.100,0.090,0.062,33.99,34.07,33.58,30.86,0,0.0,29.990,2.55,0.309,168.60,0.00
